# Lab 15: two engines, same traffic

vLLM and SGLang one after the other on random and shared-prefix traffic, with the same load generator so the client is identical. SGLang is installed into the same environment after the vLLM half; if its install conflicts with the engine already installed, the second half will say so.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
def stream_chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, "stream": True, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    t0 = time.time(); first = None; last = None; gaps = []; text = []
    with urllib.request.urlopen(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]": continue
            delta = json.loads(line[5:])["choices"][0].get("delta", {}).get("content")
            if not delta: continue
            now = time.time()
            if first is None: first = now - t0
            else: gaps.append(now - last)
            last = now; text.append(delta)
    return first, gaps, "".join(text)

def metrics(names):
    out = {}
    for line in urllib.request.urlopen(URL + "/metrics", timeout=5).read().decode().splitlines():
        if line.startswith("#"): continue
        for n in names:
            if line.startswith(n + "{") or line.startswith(n + " "):
                try: out[n] = float(line.rsplit(" ", 1)[1])
                except ValueError: pass
    return out
print("streaming and metrics helpers ready")

In [ ]:
# Cell 5: the load generator.
import random, threading
def random_prompt(n_tokens, seed):
    rng = random.Random(seed); words = ["river", "stone", "quiet", "engine", "orange", "ledger", "window", "copper", "signal", "harbor", "meadow", "lantern", "velvet", "summit", "cinder", "fable"]
    return " ".join(rng.choice(words) for _ in range(int(n_tokens * 0.8)))

def load_test(rate, num_prompts, prompt_len=512, out_len=128, prefix_len=0, model=None, concurrency=None):
    rng = random.Random(1); prefix = random_prompt(prefix_len, 99) + " " if prefix_len else ""
    prompts = [prefix + random_prompt(prompt_len - prefix_len, i) for i in range(num_prompts)]; results = []; lock = threading.Lock()
    def one(i):
        first, gaps, text = stream_chat([{"role": "user", "content": prompts[i]}], **({"model": model} if model else {}), max_tokens=out_len, temperature=0, ignore_eos=True)
        with lock: results.append((first, (sum(gaps) / len(gaps)) if gaps else 0.0, len(gaps) + 1))
    sem = threading.Semaphore(concurrency or num_prompts); t0 = time.time(); threads = []
    def gated(i):
        with sem: one(i)
    for i in range(num_prompts):
        th = threading.Thread(target=gated, args=(i,)); th.start(); threads.append(th)
        if not concurrency: time.sleep(rng.expovariate(rate))
    for th in threads: th.join()
    dur = time.time() - t0; ttft = sorted(r[0] for r in results); tpot = sorted(r[1] for r in results); q = lambda xs, p: xs[min(len(xs) - 1, int(p * len(xs)))]
    return {"req_s": round(len(results) / dur, 2), "out_tok_s": round(sum(r[2] for r in results) / dur, 1), "ttft_p50_ms": round(1000 * q(ttft, 0.5)), "ttft_p99_ms": round(1000 * q(ttft, 0.99)), "tpot_p50_ms": round(1000 * q(tpot, 0.5), 1), "tpot_p99_ms": round(1000 * q(tpot, 0.99), 1)}

def load_row(label, r):
    return f"{label:<30} req/s {r['req_s']:5.2f}  out tok/s {r['out_tok_s']:7.1f}  TTFT p50 {r['ttft_p50_ms']:6d} p99 {r['ttft_p99_ms']:6d} ms  TPOT p50 {r['tpot_p50_ms']:5.1f} p99 {r['tpot_p99_ms']:5.1f} ms"
print("load generator ready")

In [ ]:
# Cell 6: vLLM.
def traffic(engine):
    for label, prefix in (("random prompts", 0), ("75% shared prefix", 768)):
        print("  " + load_row(f"{engine}: {label}", load_test(rate=2, num_prompts=32, prompt_len=1024, out_len=128, prefix_len=prefix)))
proc, ready_s = serve(["--max-model-len", "4096"]); print(f"vLLM ready after {ready_s} s"); traffic("vLLM"); stop(proc)

In [ ]:
# Cell 7: SGLang.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sglang[all]==0.5.22"], check=False)
log = open("/kaggle/working/sglang.log", "w"); t0 = time.time()
proc = subprocess.Popen([sys.executable, "-m", "sglang.launch_server", "--model-path", MODEL, "--port", "8000", "--context-length", "4096", "--dtype", "half"], stdout=log, stderr=subprocess.STDOUT)
while time.time() - t0 < 900 and proc.poll() is None:
    try: urllib.request.urlopen(URL + "/health", timeout=2); break
    except Exception: time.sleep(0.5)
if proc.poll() is not None: print("SGLang did not start; last lines:\n" + "\n".join(open("/kaggle/working/sglang.log").read().splitlines()[-15:]))
else: print(f"SGLang ready after {time.time() - t0:.0f} s"); traffic("SGLang"); proc.terminate(); proc.wait()